In [2]:
%matplotlib inline

import os
import sys
from sys import platform

sys.path.append('../../')

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

import tiktoken

import torch
import torch.nn as nn



%load_ext autoreload
%autoreload 2

from llm_from_scratch.CH4.gpt import GPTModel
from llm_from_scratch.CH2.text_data_set import create_dataloader_v1
from llm_from_scratch.CH5.loss import calc_loss_loader
from llm_from_scratch.CH5.utils import load_weights_into_gpt, generate, text_to_token_ids, token_ids_to_text
from gpt_download import download_and_load_gpt2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
from importlib.metadata import version

pkgs=['matplotlib', 'numpy', 'tiktoken', 'torch', 'tensorflow']
for p in pkgs: print(f"{p} version: {version(p)}")

matplotlib version: 3.10.9
numpy version: 2.2.6
tiktoken version: 0.12.0
torch version: 2.12.0
tensorflow version: 2.21.0


In [3]:
main_output_dirpath='../../../../../../results'
settings, params=download_and_load_gpt2(model_size="1558M", models_dir=f"{main_output_dirpath}/gpts")
print(f"Settings: {settings}")
print(f"Parameter dictionary keys: {params.keys()}")

GPT_CONFIG_124M={'vocab_size':50257,
                 'context_length':256,
                 'emb_dim':768,
                 'n_heads':12,
                 'n_layers':12,
                 'drop_rate':0.1,
                 'qkv_bias':False}


model_configs={'gpt2-small (124M)':{'emb_dim':768, 'n_layers':12, 'n_heads':12},
               'gpt2-medium (355M)':{'emb_dim':1024,'n_layers':24,'n_heads':16},
               'gpt2-large (774M)':{'emb_dim':1280, 'n_layers':36, 'n_heads':20},
               'gpt2-xl (1558M)':{'emb_dim':1600, 'n_layers':48, 'n_heads':25}}

model_name='gpt2-xl (1558M)'
NEW_CONFIG=GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
print(f"{NEW_CONFIG=}")
NEW_CONFIG.update({'context_length':1024, 'qkv_bias':True})
print(f"{NEW_CONFIG=}")

checkpoint: 100%|███████████████████████████████████████████████████████████████████| 77.0/77.0 [00:00<00:00, 34.3kiB/s]
encoder.json: 100%|███████████████████████████████████████████████████████████████| 1.04M/1.04M [00:00<00:00, 6.18MiB/s]
hparams.json: 100%|█████████████████████████████████████████████████████████████████| 91.0/91.0 [00:00<00:00, 60.5kiB/s]
model.ckpt.data-00000-of-00001: 100%|█████████████████████████████████████████████| 6.23G/6.23G [04:05<00:00, 25.4MiB/s]
model.ckpt.index: 100%|███████████████████████████████████████████████████████████| 20.7k/20.7k [00:00<00:00, 1.78MiB/s]
model.ckpt.meta: 100%|████████████████████████████████████████████████████████████| 1.84M/1.84M [00:00<00:00, 8.90MiB/s]
vocab.bpe: 100%|████████████████████████████████████████████████████████████████████| 456k/456k [00:00<00:00, 4.15MiB/s]


Settings: {'n_vocab': 50257, 'n_ctx': 1024, 'n_embd': 1600, 'n_head': 25, 'n_layer': 48}
Parameter dictionary keys: dict_keys(['blocks', 'b', 'g', 'wpe', 'wte'])
NEW_CONFIG={'vocab_size': 50257, 'context_length': 256, 'emb_dim': 1600, 'n_heads': 25, 'n_layers': 48, 'drop_rate': 0.1, 'qkv_bias': False}
NEW_CONFIG={'vocab_size': 50257, 'context_length': 1024, 'emb_dim': 1600, 'n_heads': 25, 'n_layers': 48, 'drop_rate': 0.1, 'qkv_bias': True}


In [4]:
if any(x in platform for x in ('linux','win32')): 
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
elif platform=='darwin':  
    device=torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"{device=}")

gpt=GPTModel(NEW_CONFIG)
load_weights_into_gpt(gpt, params)
gpt.to(device)
gpt.eval();

torch.manual_seed(123)
tokenizer=tiktoken.get_encoding('gpt2')
token_ids=generate(model=gpt, idx=text_to_token_ids("Every effort moves you", tokenizer).to(device), 
                   max_new_tokens=25, context_size=NEW_CONFIG['context_length'], temperature=1.5, top_k=50, eos_id=None)
print('output text: ', token_ids_to_text(token_ids, tokenizer))

device=device(type='mps')
output text:  Every effort moves you closer to some vision at some cost (however minor) in exchange for your hard earned soul and mind; every action or


In [5]:
filepath='../the-verdict.txt'
with open(filepath, 'r', encoding='utf-8') as f: text_data=f.read()
train_ratio=0.9
split_idx=int(train_ratio*len(text_data))
train_data=text_data[:split_idx]
val_data=text_data[split_idx:]
print(f"{split_idx=}, {len(train_data)=}, {len(val_data)=}")
train_loader=create_dataloader_v1(train_data, batch_size=2, max_length=GPT_CONFIG_124M['context_length'],
                                  stride=GPT_CONFIG_124M['context_length'], drop_last=True, shuffle=True, num_workers=0)
val_loader=create_dataloader_v1(val_data, batch_size=2, max_length=GPT_CONFIG_124M['context_length'], 
                                stride=GPT_CONFIG_124M['context_length'], drop_last=False, shuffle=False, num_workers=0)
avg_train_loss=calc_loss_loader(data_loader=train_loader, model=gpt, device=device, num_batches=None)
avg_val_loss=calc_loss_loader(data_loader=val_loader, model=gpt, device=device, num_batches=None)
print(f"{avg_train_loss=:.3f}, {avg_val_loss=:.3f}")

split_idx=18431, len(train_data)=18431, len(val_data)=2048
avg_train_loss=3.305, avg_val_loss=3.120
